In [ ]:
try:
    import pyspark.sql.functions as F
    from pyspark.sql import Window
    from pyspark.sql.types import IntegerType
except ModuleNotFoundError as error:
    raise RuntimeError(
        "Este notebook precisa ser executado em um cluster Databricks com PySpark."
    ) from error

SCHEMA_ORIGEM = "bronze"
TABELA_ORIGEM = "tb_movies_info"
SCHEMA_DESTINO = "silver"
TABELA_DESTINO = "tb_info_filmes"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_DESTINO}")

In [ ]:
df_bronze = spark.table(f"{SCHEMA_ORIGEM}.{TABELA_ORIGEM}")

colunas_esperadas = {
    "id", "tconst", "title", "original_title",
    "original_language", "release_date", "runtime",
    "status", "overview", "tagline", "ingestion_datetime"
}
colunas_ausentes = colunas_esperadas.difference(df_bronze.columns)
if colunas_ausentes:
    raise ValueError(f"Colunas ausentes na origem Bronze: {sorted(colunas_ausentes)}")

# mantém a Bronze inalterada e aplica as transformações em uma cópia
df_tratado = (
    df_bronze
    .withColumn("id_filme", F.expr("try_cast(id AS INT)"))
    .withColumn("id_imdb", F.trim(F.col("tconst")))
    .withColumn("titulo", F.trim(F.col("title")))
    .withColumn("titulo_original", F.trim(F.col("original_title")))
    .withColumn("idioma_original", F.trim(F.col("original_language")))
    .withColumn("duracao_minutos", F.expr("try_cast(runtime AS INT)"))
    .withColumn("sinopse", F.trim(F.col("overview")))
    .withColumn("tagline", F.trim(F.col("tagline")))
)

In [ ]:
# normaliza caixa, espaços e hífens antes de traduzir os status
status_normalizado = F.lower(
    F.trim(F.regexp_replace(F.coalesce(F.col("status"), F.lit("")), r"[\s_-]+", " "))
)
df_tratado = df_tratado.withColumn("status_normalizado", status_normalizado)
df_tratado = df_tratado.withColumn(
    "status",
    F.when(F.col("status_normalizado") == "released", F.lit("Lançado"))
     .when(F.col("status_normalizado") == "post production", F.lit("Pós-Produção"))
     .when(F.col("status_normalizado") == "in production", F.lit("Em Produção"))
     .when(F.col("status_normalizado") == "planned", F.lit("Planejado"))
     .otherwise(F.lit("Não Informado"))
).drop("status_normalizado")

In [ ]:
# trata os formatos de data observados; valores inválidos viram NULL
data_texto = F.trim(F.col("release_date"))
data_lancamento = F.coalesce(
    F.expr("try_to_date(release_date, 'yyyy-MM-dd')"),
    F.expr("try_to_date(release_date, 'dd/MM/yyyy')"),
    F.expr("try_to_date(release_date, 'dd-MM-yyyy')"),
    F.expr("try_to_date(release_date, 'MM-dd-yyyy')")
)

df_tratado = (
    df_tratado
    .withColumn("data_lancamento", data_lancamento)
    .withColumn("ano_lancamento", F.year(F.col("data_lancamento")).cast(IntegerType()))
)

# mantém a versão mais recente de cada filme conforme a ingestão
janela_mais_recente = Window.partitionBy("id_filme").orderBy(F.col("ingestion_datetime").desc())
df_tratado = (
    df_tratado
    .withColumn("ordem_ingestao", F.row_number().over(janela_mais_recente))
    .where(F.col("id_filme").isNotNull() & (F.col("ordem_ingestao") == 1))
    .drop("ordem_ingestao", "id", "tconst", "title", "original_title",
          "original_language", "release_date", "runtime", "overview")
)

colunas_silver = [
    "id_filme", "id_imdb", "titulo", "titulo_original",
    "idioma_original", "data_lancamento", "ano_lancamento",
    "duracao_minutos", "status", "sinopse", "tagline",
    "ingestion_datetime"
]
df_silver = df_tratado.select(*colunas_silver)

In [ ]:
# grava a Silver em overwrite para permitir reprocessamento idempotente
(df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SCHEMA_DESTINO}.{TABELA_DESTINO}"
))

print(f"Tabela gravada: {SCHEMA_DESTINO}.{TABELA_DESTINO}")
display(spark.table(f"{SCHEMA_DESTINO}.{TABELA_DESTINO}").limit(10))

In [ ]:
# valida unicidade, tipos tratados e datas não interpretáveis
df_validacao = spark.table(f"{SCHEMA_DESTINO}.{TABELA_DESTINO}")
duplicados = (
    df_validacao.groupBy("id_filme").count().where(F.col("count") > 1).count()
)
datas_invalidas = (
    df_validacao.where(F.col("data_lancamento").isNull()).count()
)

if duplicados != 0:
    raise AssertionError(f"Há {duplicados} ids de filme duplicados na Silver.")

display(
    df_validacao.groupBy("status").count().orderBy(F.col("count").desc())
)
print(f"Registros Silver: {df_validacao.count()}")
print(f"Datas nulas (inválidas ou ausentes): {datas_invalidas}")
print(f"Duplicidades por id_filme: {duplicados}")

In [ ]:
SCHEMA_FINANCEIRO_ORIGEM = "bronze"
TABELA_FINANCEIRO_ORIGEM = "tb_movies_financials"
TABELA_COTACAO_ORIGEM = "tb_cotacao_dolar"
TABELA_FINANCEIRO_DESTINO = "tb_financeiro_filmes"

df_financeiro_bronze = spark.table(f"{SCHEMA_FINANCEIRO_ORIGEM}.{TABELA_FINANCEIRO_ORIGEM}")
colunas_financeiras_esperadas = {"id", "budget", "revenue", "ingestion_datetime"}
colunas_financeiras_ausentes = colunas_financeiras_esperadas.difference(df_financeiro_bronze.columns)
if colunas_financeiras_ausentes:
    raise ValueError(f"Colunas ausentes na origem Bronze: {sorted(colunas_financeiras_ausentes)}")

In [ ]:
# renomeia os campos da bronze para o padrão da camada silver sem alterar a origem
df_financeiro = (
    df_financeiro_bronze
    .withColumn("id_filme", F.expr("try_cast(id AS INT)"))
    .withColumn("orcamento_texto", F.trim(F.col("budget")))
    .withColumn("receita_texto", F.trim(F.col("revenue")))
)

# transforma marcadores de ausência em NULL para evitar que textos contaminem os cálculos
marcadores_ausencia = r"^(|unknown|não informado|na|n/a|null)$"
df_financeiro = (
    df_financeiro
    .withColumn("orcamento_texto", F.when(F.lower(F.col("orcamento_texto")).rlike(marcadores_ausencia), F.lit(None)).otherwise(F.col("orcamento_texto")))
    .withColumn("receita_texto", F.when(F.lower(F.col("receita_texto")).rlike(marcadores_ausencia), F.lit(None)).otherwise(F.col("receita_texto")))
)

# remove símbolos monetários e separadores antes do cast; entradas inválidas são convertidas para NULL
df_financeiro = (
    df_financeiro
    .withColumn("orcamento_limpo", F.regexp_replace(F.col("orcamento_texto"), r"[$,\s]", ""))
    .withColumn("receita_limpa", F.regexp_replace(F.col("receita_texto"), r"[$,\s]", ""))
    .withColumn("orcamento_usd", F.expr("try_cast(orcamento_limpo AS DECIMAL(20,2))"))
    .withColumn("receita_usd", F.expr("try_cast(receita_limpa AS DECIMAL(20,2))"))
)

# valores nulos, zerados ou negativos não representam um valor financeiro válido
df_financeiro = (
    df_financeiro
    .withColumn("orcamento_usd", F.when(F.col("orcamento_usd") > 0, F.col("orcamento_usd")))
    .withColumn("receita_usd", F.when(F.col("receita_usd") > 0, F.col("receita_usd")))
)

In [ ]:
df_cotacao = (
    spark.table(f"{SCHEMA_FINANCEIRO_ORIGEM}.{TABELA_COTACAO_ORIGEM}")
    .withColumn("data_hora_cotacao", F.to_timestamp("dataHoraCotacao"))
)

# como a origem financeira não possui data da transação, usa-se a cotação PTAX mais recente disponível
df_cotacao_atual = (
    df_cotacao
    .where(F.col("cotacaoCompra") > 0)
    .orderBy(F.col("data_hora_cotacao").desc())
    .limit(1)
    .select(F.col("cotacaoCompra").cast("DECIMAL(12,6)").alias("cotacao_dolar_brl"))
)

# o cruzamento aplica a mesma taxa de referência a cada filme e mantém o cálculo no cluster
df_financeiro = (
    df_financeiro
    .crossJoin(df_cotacao_atual)
    .withColumn("orcamento_brl", (F.col("orcamento_usd") * F.col("cotacao_dolar_brl")).cast("DECIMAL(20,2)"))
    .withColumn("receita_brl", (F.col("receita_usd") * F.col("cotacao_dolar_brl")).cast("DECIMAL(20,2)"))
    .withColumn("lucro_usd", (F.col("receita_usd") - F.col("orcamento_usd")).cast("DECIMAL(20,2)"))
    .withColumn("lucro_brl", (F.col("receita_brl") - F.col("orcamento_brl")).cast("DECIMAL(20,2)"))
    .withColumn("margem_lucro_percentual", F.when(F.col("receita_usd") > 0, (F.col("lucro_usd") / F.col("receita_usd") * 100).cast("DECIMAL(10,2)")))
)

In [ ]:
colunas_financeiro_silver = [
    "id_filme", "orcamento_usd", "receita_usd",
    "cotacao_dolar_brl", "orcamento_brl", "receita_brl",
    "lucro_usd", "lucro_brl", "margem_lucro_percentual",
    "ingestion_datetime"
]

# mantém somente a versão mais recente de cada filme após as cargas append da bronze
janela_financeira_mais_recente = Window.partitionBy("id_filme").orderBy(F.col("ingestion_datetime").desc())
df_financeiro_silver = (
    df_financeiro
    .where(F.col("id_filme").isNotNull())
    .withColumn("ordem_ingestao", F.row_number().over(janela_financeira_mais_recente))
    .where(F.col("ordem_ingestao") == 1)
    .drop("ordem_ingestao")
    .select(*colunas_financeiro_silver)
)

# grava em overwrite para permitir reprocessamento idempotente da tabela silver
(df_financeiro_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SCHEMA_DESTINO}.{TABELA_FINANCEIRO_DESTINO}"
))

print(f"Tabela gravada: {SCHEMA_DESTINO}.{TABELA_FINANCEIRO_DESTINO}")
display(spark.table(f"{SCHEMA_DESTINO}.{TABELA_FINANCEIRO_DESTINO}").limit(10))

In [ ]:
# valida identificadores, conversões financeiras e ausência de divisão por receita nula
df_financeiro_validacao = spark.table(f"{SCHEMA_DESTINO}.{TABELA_FINANCEIRO_DESTINO}")
duplicados_financeiros = (
    df_financeiro_validacao.groupBy("id_filme").count().where(F.col("count") > 1).count()
)
if duplicados_financeiros != 0:
    raise AssertionError(f"Há {duplicados_financeiros} ids de filme duplicados na Silver financeira.")

display(df_financeiro_validacao.select(
    "id_filme", "orcamento_usd", "receita_usd", "lucro_usd", "margem_lucro_percentual"
).limit(10))
print(f"Registros Silver financeira: {df_financeiro_validacao.count()}")
print(f"Orçamentos válidos: {df_financeiro_validacao.where(F.col("orcamento_usd").isNotNull()).count()}")
print(f"Receitas válidas: {df_financeiro_validacao.where(F.col("receita_usd").isNotNull()).count()}")
print(f"Registros sem receita válida: {df_financeiro_validacao.where(F.col("receita_usd").isNull()).count()}")
print(f"Registros sem orçamento válido: {df_financeiro_validacao.where(F.col("orcamento_usd").isNull()).count()}")
print(f"Duplicidades por id_filme: {duplicados_financeiros}")

In [ ]:
SCHEMA_METRICAS_ORIGEM = "bronze"
TABELA_METRICAS_ORIGEM = "tb_movies_metrics"
TABELA_METRICAS_DESTINO = "tb_metricas_engajamento"

df_metricas_bronze = spark.table(f"{SCHEMA_METRICAS_ORIGEM}.{TABELA_METRICAS_ORIGEM}")
colunas_metricas_esperadas = {"id", "popularity", "vote_average", "vote_count", "averageRating", "numVotes", "ingestion_datetime"}
colunas_metricas_ausentes = colunas_metricas_esperadas.difference(df_metricas_bronze.columns)
if colunas_metricas_ausentes:
    raise ValueError(f"Colunas ausentes na origem Bronze: {sorted(colunas_metricas_ausentes)}")

In [ ]:
# renomeia os campos para o padrão da silver sem alterar a tabela bronze
df_metricas = (
    df_metricas_bronze
    .withColumn("id_filme", F.expr("try_cast(id AS INT)"))
    .withColumn("popularidade_texto", F.trim(F.col("popularity")))
    .withColumn("nota_media_tmdb_texto", F.trim(F.col("vote_average")))
    .withColumn("quantidade_votos_tmdb_texto", F.trim(F.col("vote_count")))
    .withColumn("nota_media_imdb_texto", F.trim(F.col("averageRating")))
    .withColumn("quantidade_votos_imdb_texto", F.trim(F.col("numVotes")))
)

# a origem mistura ponto decimal e vírgula decimal; remove pontuação de milhar antes da conversão
popularidade_normalizada = F.when(
    F.col("popularidade_texto").contains(","),
    F.regexp_replace(
        F.regexp_replace(F.col("popularidade_texto"), r"\.", ""),
        ",", "."
    )
).otherwise(F.col("popularidade_texto"))

df_metricas = (
    df_metricas
    .withColumn("popularidade_normalizada", popularidade_normalizada)
    .withColumn("popularidade", F.expr("try_cast(popularidade_normalizada AS DOUBLE)"))
    .withColumn("nota_media_tmdb", F.expr("try_cast(nota_media_tmdb_texto AS DOUBLE)"))
    .withColumn("quantidade_votos_tmdb", F.expr("try_cast(quantidade_votos_tmdb_texto AS BIGINT)"))
    .withColumn("nota_media_imdb", F.expr("try_cast(nota_media_imdb_texto AS DOUBLE)"))
    .withColumn("quantidade_votos_imdb", F.expr("try_cast(quantidade_votos_imdb_texto AS BIGINT)"))
)

# valores fora do domínio sinalizam column shift ou sujeira e não devem contaminar as métricas
df_metricas = (
    df_metricas
    .withColumn("popularidade", F.when(F.col("popularidade") >= 0, F.col("popularidade")))
    .withColumn("quantidade_votos_tmdb", F.when(F.col("quantidade_votos_tmdb") >= 0, F.col("quantidade_votos_tmdb")))
    .withColumn("quantidade_votos_imdb", F.when(F.col("quantidade_votos_imdb") >= 0, F.col("quantidade_votos_imdb")))
    .withColumn("nota_media_tmdb", F.when(F.col("nota_media_tmdb").between(0, 10), F.col("nota_media_tmdb")))
    .withColumn("nota_media_imdb", F.when(F.col("nota_media_imdb").between(0, 10), F.col("nota_media_imdb")))
)

In [ ]:
colunas_metricas_silver = [
    "id_filme", "popularidade", "nota_media_tmdb",
    "quantidade_votos_tmdb", "nota_media_imdb",
    "quantidade_votos_imdb", "ingestion_datetime"
]

# como a bronze é append-only, mantém a versão mais recente de cada filme
janela_metricas_mais_recente = Window.partitionBy("id_filme").orderBy(F.col("ingestion_datetime").desc())
df_metricas_silver = (
    df_metricas
    .where(F.col("id_filme").isNotNull())
    .withColumn("ordem_ingestao", F.row_number().over(janela_metricas_mais_recente))
    .where(F.col("ordem_ingestao") == 1)
    .drop("ordem_ingestao", "id", "popularity", "vote_average",
          "vote_count", "averageRating", "numVotes",
          "popularidade_texto", "popularidade_normalizada",
          "nota_media_tmdb_texto", "quantidade_votos_tmdb_texto",
          "nota_media_imdb_texto", "quantidade_votos_imdb_texto")
    .select(*colunas_metricas_silver)
)

# grava em overwrite para permitir reprocessamento idempotente da tabela Silver
(df_metricas_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SCHEMA_DESTINO}.{TABELA_METRICAS_DESTINO}"
))

print(f"Tabela gravada: {SCHEMA_DESTINO}.{TABELA_METRICAS_DESTINO}")
display(spark.table(f"{SCHEMA_DESTINO}.{TABELA_METRICAS_DESTINO}").limit(10))

In [ ]:
# valida limites das notas, valores não negativos e unicidade por filme
df_metricas_validacao = spark.table(f"{SCHEMA_DESTINO}.{TABELA_METRICAS_DESTINO}")
duplicados_metricas = (
    df_metricas_validacao.groupBy("id_filme").count().where(F.col("count") > 1).count()
)
violacoes_notas = df_metricas_validacao.where(
    F.col("nota_media_tmdb").isNotNull() & ~F.col("nota_media_tmdb").between(0, 10)
).count() + df_metricas_validacao.where(
    F.col("nota_media_imdb").isNotNull() & ~F.col("nota_media_imdb").between(0, 10)
).count()

if duplicados_metricas != 0:
    raise AssertionError(f"Há {duplicados_metricas} ids de filme duplicados na Silver de métricas.")
if violacoes_notas != 0:
    raise AssertionError(f"Há {violacoes_notas} notas fora do intervalo permitido.")

display(df_metricas_validacao.select(
    "id_filme", "popularidade", "nota_media_tmdb",
    "quantidade_votos_tmdb", "nota_media_imdb", "quantidade_votos_imdb"
).limit(10))
print(f"Registros Silver de métricas: {df_metricas_validacao.count()}")
print(f"Popularidades nulas ou inválidas: {df_metricas_validacao.where(F.col("popularidade").isNull()).count()}")
print(f"Notas nulas ou inválidas: {df_metricas_validacao.where(F.col("nota_media_tmdb").isNull() | F.col("nota_media_imdb").isNull()).count()}")
print(f"Duplicidades por id_filme: {duplicados_metricas}")

In [ ]:
SCHEMA_AVALIACOES_ORIGEM = "bronze"
TABELA_AVALIACOES_ORIGEM = "tb_movies_reviews"
TABELA_AVALIACOES_DESTINO = "tb_avaliacoes_usuarios"

df_avaliacoes_bronze = spark.table(f"{SCHEMA_AVALIACOES_ORIGEM}.{TABELA_AVALIACOES_ORIGEM}")
colunas_avaliacoes_esperadas = {"id", "nome", "nota", "comentario", "ingestion_datetime"}
colunas_avaliacoes_ausentes = colunas_avaliacoes_esperadas.difference(df_avaliacoes_bronze.columns)
if colunas_avaliacoes_ausentes:
    raise ValueError(f"Colunas ausentes na origem Bronze: {sorted(colunas_avaliacoes_ausentes)}")

In [ ]:
# renomeia os campos e normaliza textos sem alterar a tabela Bronze
df_avaliacoes = (
    df_avaliacoes_bronze
    .withColumn("id_filme", F.expr("try_cast(id AS INT)"))
    .withColumn("nome_usuario", F.trim(F.col("nome")))
    .withColumn("nota", F.expr("try_cast(nota AS DOUBLE)"))
    .withColumn("comentario", F.trim(F.col("comentario")))
)

# notas fora do domínio de avaliação são inválidas e devem ficar como NULL
df_avaliacoes = df_avaliacoes.withColumn(
    "nota",
    F.when(F.col("nota").between(0, 10), F.col("nota"))
)

# comentários nulos ou somente com espaços recebem um texto padrão
df_avaliacoes = df_avaliacoes.withColumn(
    "comentario",
    F.when(F.col("comentario").isNull() | (F.col("comentario") == ""), F.lit("Sem comentário"))
     .otherwise(F.col("comentario"))
)

# remove duplicatas integrais da avaliação, desconsiderando o timestamp de ingestão
colunas_chave_avaliacao = ["id_filme", "nome_usuario", "nota", "comentario"]
colunas_avaliacoes_silver = colunas_chave_avaliacao + ["ingestion_datetime"]
df_avaliacoes_silver = (
    df_avaliacoes
    .dropDuplicates(colunas_chave_avaliacao)
    .select(*colunas_avaliacoes_silver)
)

In [ ]:
# grava em overwrite para permitir reprocessamento idempotente da tabela Silver
(df_avaliacoes_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SCHEMA_DESTINO}.{TABELA_AVALIACOES_DESTINO}"
))

print(f"Tabela gravada: {SCHEMA_DESTINO}.{TABELA_AVALIACOES_DESTINO}")
display(spark.table(f"{SCHEMA_DESTINO}.{TABELA_AVALIACOES_DESTINO}").limit(10))

In [ ]:
# valida notas, comentários preenchidos e ausência de duplicatas integrais
df_avaliacoes_validacao = spark.table(f"{SCHEMA_DESTINO}.{TABELA_AVALIACOES_DESTINO}")
duplicados_avaliacoes = (
    df_avaliacoes_validacao.groupBy(*colunas_chave_avaliacao).count().where(F.col("count") > 1).count()
)
violacoes_notas_avaliacoes = df_avaliacoes_validacao.where(
    F.col("nota").isNotNull() & ~F.col("nota").between(0, 10)
).count()
comentarios_vazios = df_avaliacoes_validacao.where(
    F.col("comentario").isNull() | (F.trim(F.col("comentario")) == "")
).count()

if duplicados_avaliacoes != 0:
    raise AssertionError(f"Há {duplicados_avaliacoes} avaliações duplicadas na Silver.")
if violacoes_notas_avaliacoes != 0:
    raise AssertionError(f"Há {violacoes_notas_avaliacoes} notas fora do intervalo permitido.")
if comentarios_vazios != 0:
    raise AssertionError(f"Há {comentarios_vazios} comentários vazios na Silver.")

display(df_avaliacoes_validacao.select(*colunas_avaliacoes_silver).limit(10))
print(f"Registros Silver de avaliações: {df_avaliacoes_validacao.count()}")
print(f"Notas nulas ou inválidas: {df_avaliacoes_validacao.where(F.col('nota').isNull()).count()}")
print(f"Comentários padronizados: {df_avaliacoes_validacao.where(F.col('comentario') == 'Sem comentário').count()}")
print(f"Duplicidades integrais: {duplicados_avaliacoes}")

In [ ]:
SCHEMA_GENEROS_ORIGEM = "bronze"
TABELA_GENEROS_ORIGEM = "tb_credits_and_tags"
TABELA_GENEROS_DESTINO = "tb_generos"

df_generos_bronze = spark.table(f"{SCHEMA_GENEROS_ORIGEM}.{TABELA_GENEROS_ORIGEM}")
colunas_generos_esperadas = {"id", "genres", "ingestion_datetime"}
colunas_generos_ausentes = colunas_generos_esperadas.difference(df_generos_bronze.columns)
if colunas_generos_ausentes:
    raise ValueError(f"Colunas ausentes na origem Bronze: {sorted(colunas_generos_ausentes)}")

In [ ]:
# normaliza separadores para que listas com vírgula, ponto e vírgula ou barra vertical tenham o mesmo tratamento
df_generos = (
    df_generos_bronze
    .withColumn("id_filme", F.expr("try_cast(id AS INT)"))
    .withColumn("generos_texto", F.trim(F.coalesce(F.col("genres"), F.lit(""))))
    .withColumn("generos_normalizados", F.regexp_replace(F.col("generos_texto"), r"\s*[;|]\s*", ","))
)

# explode transforma a lista em relações filme-gênero sem alterar a tabela bronze
df_generos = (
    df_generos
    .withColumn("genero_bruto", F.explode(F.split(F.col("generos_normalizados"), ",")))
    .withColumn("nome_genero", F.trim(F.regexp_replace(F.col("genero_bruto"), r"[\[\]{}\"]", "")))
)

# remove marcadores de ausência e resíduos que não representam um gênero válido
marcadores_genero_ausente = ["", "n/a", "na", "null", "unknown", "não informado", "[]"]
df_generos = (
    df_generos
    .withColumn("nome_genero", F.trim(F.col("nome_genero")))
    .where(F.col("id_filme").isNotNull())
    .where(F.length(F.col("nome_genero")) > 0)
    .where(~F.lower(F.col("nome_genero")).isin(marcadores_genero_ausente))
)

In [ ]:
# como a bronze é append-only, conserva a versão mais recente de cada relação filme-gênero
janela_genero_mais_recente = Window.partitionBy("id_filme", "nome_genero").orderBy(F.col("ingestion_datetime").desc())
colunas_generos_silver = ["id_filme", "nome_genero", "ingestion_datetime"]
df_generos_silver = (
    df_generos
    .withColumn("ordem_ingestao", F.row_number().over(janela_genero_mais_recente))
    .where(F.col("ordem_ingestao") == 1)
    .select(*colunas_generos_silver)
)

# grava em overwrite para permitir reprocessamento idempotente da tabela silver
(df_generos_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SCHEMA_DESTINO}.{TABELA_GENEROS_DESTINO}"
))

print(f"Tabela gravada: {SCHEMA_DESTINO}.{TABELA_GENEROS_DESTINO}")
display(spark.table(f"{SCHEMA_DESTINO}.{TABELA_GENEROS_DESTINO}").limit(10))

In [ ]:
# valida que cada linha contém um gênero limpo e que a relação filme-gênero não se repete
df_generos_validacao = spark.table(f"{SCHEMA_DESTINO}.{TABELA_GENEROS_DESTINO}")
duplicados_generos = (
    df_generos_validacao.groupBy("id_filme", "nome_genero").count().where(F.col("count") > 1).count()
)
generos_vazios = df_generos_validacao.where(
    F.col("nome_genero").isNull() | (F.trim(F.col("nome_genero")) == "")
).count()

if duplicados_generos != 0:
    raise AssertionError(f"Há {duplicados_generos} relações filme-gênero duplicadas na Silver.")
if generos_vazios != 0:
    raise AssertionError(f"Há {generos_vazios} gêneros vazios na Silver.")

display(df_generos_validacao.groupBy("nome_genero").count().orderBy(F.col("count").desc()))
print(f"Registros Silver de gêneros: {df_generos_validacao.count()}")
print(f"Filmes com gênero: {df_generos_validacao.select('id_filme').distinct().count()}")
print(f"Duplicidades filme-gênero: {duplicados_generos}")

In [ ]:
SCHEMA_PESSOAS_ORIGEM = "bronze"
TABELA_PESSOAS_ORIGEM = "tb_credits_and_tags"
TABELA_PESSOAS_DESTINO = "tb_pessoas_empresas"

df_pessoas_bronze = spark.table(f"{SCHEMA_PESSOAS_ORIGEM}.{TABELA_PESSOAS_ORIGEM}")
colunas_pessoas_esperadas = {"id", "cast", "directors", "writers", "production_companies", "ingestion_datetime"}
colunas_pessoas_ausentes = colunas_pessoas_esperadas.difference(df_pessoas_bronze.columns)
if colunas_pessoas_ausentes:
    raise ValueError(f"Colunas ausentes na origem Bronze: {sorted(colunas_pessoas_ausentes)}")

In [ ]:
# normaliza listas, marcadores de ausência e identificadores sem alterar a bronze
df_pessoas = (
    df_pessoas_bronze
    .withColumn("id_filme", F.expr("try_cast(id AS INT)"))
    .withColumn("atores_texto", F.trim(F.coalesce(F.col("cast"), F.lit(""))))
    .withColumn("diretores_texto", F.trim(F.coalesce(F.col("directors"), F.lit(""))))
    .withColumn("roteiristas_texto", F.trim(F.coalesce(F.col("writers"), F.lit(""))))
    .withColumn("produtoras_texto", F.trim(F.coalesce(F.col("production_companies"), F.lit(""))))
)

# unpivot consolida as quatro origens em uma dimensão com o tipo da entidade
df_pessoas = df_pessoas.select(
    "id_filme", "ingestion_datetime",
    F.expr("stack(4, 'Ator', atores_texto, 'Diretor', diretores_texto, 'Roteirista', roteiristas_texto, 'Produtora', produtoras_texto) AS (tipo_entidade, entidades_texto)")
)

# separa listas antes do explode e padroniza a capitalização dos nomes
marcadores_entidade_ausente = ["", "n/a", "na", "null", "unknown", "não informado", "[]"]
df_pessoas = (
    df_pessoas
    .withColumn("entidade_bruta", F.explode(F.split(F.col("entidades_texto"), r"\s*[,;|]\s*")))
    .withColumn("nome_entidade", F.initcap(F.trim(F.regexp_replace(F.col("entidade_bruta"), r"[\[\]{}\"]", ""))))
    .where(F.col("id_filme").isNotNull())
    .where(F.length(F.col("nome_entidade")) > 0)
    .where(~F.lower(F.col("nome_entidade")).isin(marcadores_entidade_ausente))
)

# mantém a versão mais recente e remove entidades repetidas no mesmo filme e tipo
janela_pessoa_mais_recente = Window.partitionBy("id_filme", "nome_entidade", "tipo_entidade").orderBy(F.col("ingestion_datetime").desc())
colunas_pessoas_silver = ["id_filme", "nome_entidade", "tipo_entidade", "ingestion_datetime"]
df_pessoas_silver = (
    df_pessoas
    .withColumn("ordem_ingestao", F.row_number().over(janela_pessoa_mais_recente))
    .where(F.col("ordem_ingestao") == 1)
    .select(*colunas_pessoas_silver)
)

In [ ]:
# grava em overwrite para permitir reprocessamento idempotente da tabela silver
(df_pessoas_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SCHEMA_DESTINO}.{TABELA_PESSOAS_DESTINO}"
))

print(f"Tabela gravada: {SCHEMA_DESTINO}.{TABELA_PESSOAS_DESTINO}")
display(spark.table(f"{SCHEMA_DESTINO}.{TABELA_PESSOAS_DESTINO}").limit(10))

In [ ]:
# valida tipos permitidos, nomes preenchidos e ausência de duplicatas
df_pessoas_validacao = spark.table(f"{SCHEMA_DESTINO}.{TABELA_PESSOAS_DESTINO}")
duplicados_pessoas = (
    df_pessoas_validacao.groupBy("id_filme", "nome_entidade", "tipo_entidade").count().where(F.col("count") > 1).count()
)
tipos_invalidos = df_pessoas_validacao.where(
    ~F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista", "Produtora")
).count()
entidades_vazias = df_pessoas_validacao.where(
    F.col("nome_entidade").isNull() | (F.trim(F.col("nome_entidade")) == "")
).count()

if duplicados_pessoas != 0:
    raise AssertionError(f"Há {duplicados_pessoas} entidades duplicadas na Silver.")
if tipos_invalidos != 0:
    raise AssertionError(f"Há {tipos_invalidos} tipos de entidade inválidos na Silver.")
if entidades_vazias != 0:
    raise AssertionError(f"Há {entidades_vazias} entidades vazias na Silver.")

display(df_pessoas_validacao.groupBy("tipo_entidade").count().orderBy(F.col("count").desc()))
print(f"Registros Silver de pessoas e empresas: {df_pessoas_validacao.count()}")
print(f"Filmes relacionados: {df_pessoas_validacao.select('id_filme').distinct().count()}")
print(f"Duplicidades filme-entidade-tipo: {duplicados_pessoas}")

In [ ]:
SCHEMA_COTACAO_ORIGEM = "bronze"
TABELA_COTACAO_ORIGEM = "tb_cotacao_dolar"
TABELA_COTACAO_DESTINO = "tb_cotacao_dolar"

df_cotacao_bronze = spark.table(f"{SCHEMA_COTACAO_ORIGEM}.{TABELA_COTACAO_ORIGEM}")
colunas_cotacao_esperadas = {"dataHoraCotacao", "cotacaoCompra", "ingestion_datetime"}
colunas_cotacao_ausentes = colunas_cotacao_esperadas.difference(df_cotacao_bronze.columns)
if colunas_cotacao_ausentes:
    raise ValueError(f"Colunas ausentes na origem Bronze: {sorted(colunas_cotacao_ausentes)}")

In [ ]:
# converte o horário da API e mantém somente cotações válidas
df_cotacao_diaria = (
    df_cotacao_bronze
    .withColumn("data_hora_cotacao", F.to_timestamp("dataHoraCotacao"))
    .withColumn("data_cotacao", F.to_date("data_hora_cotacao"))
    .withColumn("cotacao_dolar_brl", F.col("cotacaoCompra").cast("DECIMAL(12,6)"))
    .where(F.col("data_cotacao").isNotNull())
    .where(F.col("cotacao_dolar_brl") > 0)
)

# conserva a última cotação disponível em cada dia
janela_cotacao_dia = Window.partitionBy("data_cotacao").orderBy(
    F.col("data_hora_cotacao").desc(),
    F.col("ingestion_datetime").desc(),
)
df_cotacao_diaria = (
    df_cotacao_diaria
    .withColumn("ordem_cotacao", F.row_number().over(janela_cotacao_dia))
    .where(F.col("ordem_cotacao") == 1)
    .select("data_cotacao", "cotacao_dolar_brl", "ingestion_datetime")
)

# cria o calendário completo entre a primeira e a última cotação
limites_cotacao = df_cotacao_diaria.agg(
    F.min("data_cotacao").alias("data_minima"),
    F.max("data_cotacao").alias("data_maxima"),
).first()
if limites_cotacao["data_minima"] is None:
    raise ValueError("A origem Bronze não possui cotações válidas.")

df_calendario_cotacao = spark.range(1).select(
    F.explode(
        F.sequence(F.lit(limites_cotacao["data_minima"]), F.lit(limites_cotacao["data_maxima"]))
    ).alias("data_cotacao")
)

# preenche fins de semana e feriados com a última cotação útil conhecida
janela_forward_fill = Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, Window.currentRow)
df_cotacao_silver = (
    df_calendario_cotacao
    .join(df_cotacao_diaria, on="data_cotacao", how="left")
    .withColumn("cotacao_dolar_brl", F.last("cotacao_dolar_brl", ignorenulls=True).over(janela_forward_fill))
    .withColumn("ingestion_datetime", F.last("ingestion_datetime", ignorenulls=True).over(janela_forward_fill))
    .select("data_cotacao", "cotacao_dolar_brl", "ingestion_datetime")
    .orderBy("data_cotacao")
)

In [ ]:
# grava em overwrite para permitir reprocessamento idempotente da tabela silver
(df_cotacao_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SCHEMA_DESTINO}.{TABELA_COTACAO_DESTINO}"
))

print(f"Tabela gravada: {SCHEMA_DESTINO}.{TABELA_COTACAO_DESTINO}")
display(spark.table(f"{SCHEMA_DESTINO}.{TABELA_COTACAO_DESTINO}").orderBy("data_cotacao").limit(10))

In [ ]:
# valida continuidade das datas, preenchimento das cotações e ausência de duplicatas
df_cotacao_validacao = spark.table(f"{SCHEMA_DESTINO}.{TABELA_COTACAO_DESTINO}")
duplicados_cotacao = df_cotacao_validacao.groupBy("data_cotacao").count().where(F.col("count") > 1).count()
datas_cotacao = df_cotacao_validacao.agg(
    F.min("data_cotacao").alias("data_minima"),
    F.max("data_cotacao").alias("data_maxima"),
    F.count("data_cotacao").alias("quantidade_datas"),
).first()
quantidade_datas_esperada = (datas_cotacao["data_maxima"] - datas_cotacao["data_minima"]).days + 1
cotacoes_vazias = df_cotacao_validacao.where(F.col("cotacao_dolar_brl").isNull()).count()

if duplicados_cotacao != 0:
    raise AssertionError(f"Há {duplicados_cotacao} datas de cotação duplicadas na Silver.")
if datas_cotacao["quantidade_datas"] != quantidade_datas_esperada:
    raise AssertionError("O calendário de cotações possui datas faltantes.")
if cotacoes_vazias != 0:
    raise AssertionError(f"Há {cotacoes_vazias} datas sem cotação preenchida na Silver.")

display(df_cotacao_validacao.orderBy("data_cotacao").limit(10))
print(f"Registros Silver de cotação: {datas_cotacao['quantidade_datas']}")
print(f"Datas sem cotação: {cotacoes_vazias}")
print(f"Duplicidades por data: {duplicados_cotacao}")